# Database Design — High-Level Design (HLD)

**System:** File Validation & Routing Framework (`fvr` database, MongoDB) • **Version:** 1.0 • **Status:** DRAFT FOR REVIEW • **Companion LLD:** `Database Design LLD — File Validation & Routing (MongoDB)`

## 1. Purpose & Scope

This HLD defines the **database architecture** backing the framework: a single MongoDB database holding all framework metadata (21 collections) plus the physical files (GridFS). It covers structure, storage, deployment topology, security, availability/DR, and scalability. Field-level definitions, validators, and enums live in the LLD.

Scope covers all phases — Phase 1 (UI Portal) through Phase 3 (Databricks reconciliation + archival) — so the database never gets redesigned when new sources arrive.

## 2. References

* `File Validation & Routing Framework — Design & Specs.ipynb` — v1.3 master design (architecture + full mongosh validator/index specs)
* `UI Portal Data Model — Phase 1 Plan (MongoDB & GridFS).ipynb` — portal-first cut, seeds, open questions
* `README.md` — project overview + repo layout

## 3. System Context

```
┌─────────────┐     HTTPS/JSON      ┌──────────────┐        ┌───────────────────────┐
│ React Portal │ ──────────────────► │   FastAPI     │ ─────► │  MongoDB (fvr)        │
│ upload/ops/ │ ◄────────────────── │ (SOLE WRITER) │ ◄───── │ • 21 metadata coll.    │
│ admin        │                    └──────┬───────┘        │ • GridFS (file bytes)  │
└─────────────┘                           │                 └───────────┬───────────┘
                                          │ download (GridFS)           │ change streams
                                          ▼                             ▼
                        ┌────────────────────────┐      ┌──────────────────────┐
                        │ Databricks Intake /    │      │ FastAPI event worker │
                        │ Quarantine volumes     │      │ (validation/routing) │
                        │ (per-file-type folders) │      └──────────────────────┘
                        └───────────┬────────────┘
                                    ▼  Auto Loader → bronze/silver/gold
                        ┌────────────────────────┐
                        │ Databricks pushback job │ ── POST /internal/ingestion-results ─► FastAPI ─► MongoDB
                        └────────────────────────┘
```

**Golden rules:** FastAPI is the only writer to MongoDB (single validation + audit surface). The React client never connects to Mongo. GridFS holds bytes only — `file_registry` is the single source of truth for state.

## 4. Data Architecture — Layers & Collections

21 collections in 5 layers (aggregate-oriented design: one control document per file, enriched by per-rule results and per-transfer manifests; configuration as versioned documents):

| Layer | Collections | Role |
| --- | --- | --- |
| **Registry** (8) | `users`, `source_systems`, `file_types`, `schema_versions`, `validation_rules`, `routing_policies`, `volume_map`, `retention_policies` | Versioned, metadata-driven configuration: file contracts, rules, routing, destinations, retention — changing behavior = inserting/updating documents, not deploying code |
| **Eventing** (1) | `event_log` | Canonical event store + queue (change streams); unique `(event_type, dedup_key)` makes at-least-once delivery safe |
| **Execution** (5) | `upload_sessions`, `file_registry`, `validation_results`, `batches`, `file_transfers` | The runtime state: who uploaded what, the per-file control entity + state machine, rule outcomes with evidence, and the GridFS→volume transfer manifest |
| **Governance** (4) | `audit_log`, `file_lifecycle`, `notifications`, `archive_jobs` | Append-only audit trail, physical location history, alert routing, retention execution |
| **Resilience** (3) | `databricks_ingestion_results`, `reconciliation_runs`, `reconciliation_results` | Closes the loop with Databricks; certifies uploaded = routed = transferred = bronze-processed; gates GridFS purge |

**Document model rationale:** MongoDB was chosen over a relational engine because (a) rule `params`, schema `fields[]`, DQ `dq_results` and audit `before/after` are naturally free-form JSON; (b) contracts and rules must be versioned data, not DDL migrations; (c) files and metadata live in one system with transactional session updates; (d) change streams provide the event backbone without extra infrastructure.

**Relationship spine:** `file_id` joins a file's entire life (`file_registry` ← `validation_results`, `file_transfers`, `file_lifecycle`, `databricks_ingestion_results`, `reconciliation_results`); `upload_session_id` groups files; `file_type_id` links contracts (`schema_versions`, `validation_rules.applies_to`, `volume_map`); `correlation_id` ties every artifact back to the originating event. References are application-enforced (validators) — MongoDB has no FK constraints.

**Phase map:** Phase 1 builds 14 collections (registry 7 + `users`, `event_log`, execution 4, governance 2, notifications lite); Phase 2 adds `batches` + remaining sources' adapters; Phase 3 adds resilience 3 + `retention_policies`/`archive_jobs`. No collection changes shape between phases.

## 5. Storage Architecture & Topology

### Bytes vs metadata

* **GridFS (single default bucket `fs`)** — all file bytes regardless of type/source; 255 KB chunks (raise to 1 MB when typical files exceed ~100 MB). Chunking gives resumable uploads for free. `fs.files` metadata is minimal reverse-lookup only (`file_id`, `sha256`, `original_name`, `upload_session_id`) — never framework state.
* **Metadata collections** — small, indexed, high-read documents; the sole source of truth for state and configuration.

### Deployment topology

* **Replica set (3 nodes minimum)** — mandatory for change streams and HA: 1 primary + 2 secondaries.
* **Atlas vs self-managed** is an open decision (Atlas preferred for PITR backups, managed upgrades; self-managed if corporate hosting policy requires it).
* **Writers:** FastAPI app user (read-write on `fvr`). **Readers:** read-only analytics user for dashboards/exports. No other principals.

### Growth drivers

| Store | Driver | Est. growth per 1,000 files/day |
| --- | --- | --- |
| GridFS | file bytes (e.g. 20 MB avg) | ~20 GB/day |
| `file_registry` | 1 doc/file (~1–2 KB) | ~1k docs/day |
| `validation_results` | ~8 docs/file (1/rule) | ~8k docs/day |
| `event_log` | ~12–15 events/file | ~15k docs/day — TTL prunes `PROCESSED` (30 d), `DEAD` never expires |
| `file_lifecycle` | 2–3 docs/file | ~3k docs/day |

Retention: Phase 1 keeps everything (purge arrives with Phase 3 reconciliation gates). The `event_log` TTL is the only Phase-1 pruning.

## 6. Key Design Decisions

| # | Decision | Rationale | Rejected alternative |
| --- | --- | --- | --- |
| D1 | MongoDB as sole metadata store + GridFS for bytes | One system for state + files; JSON-native rule params/schema fields/DQ results; change streams for events without extra infra | RDBMS + object store (schema migrations per contract change; two systems to reconcile) |
| D2 | Single GridFS bucket for all file types | Cross-type duplicate detection (one sha256 unique index); simple ops; per-type separation happens at routing time via `volume_map` | One bucket per file type (splits dedup scope, complicates ops) |
| D3 | Idempotency via unique indexes, not app logic | `file_registry.sha256_checksum` unique, `event_log (event_type, dedup_key)` unique, partial-unique `file_transfers (file_id, direction) WHERE COMPLETED` — duplicate deliveries/retries are structurally impossible to double-apply | Application-level "check-then-insert" (race-prone) |
| D4 | Versioned contracts (`schema_version_id`, `rule_version`, `effective_from`) | Every historical result is reproducible against the exact contract that produced it; changes never break history | In-place edits (no auditability) |
| D5 | Canonical event envelope from day one | Portal emits `FILE_ARRIVED` exactly as Phase-2 adapters will — the processing loop is source-agnostic from the first deploy | Portal-only synchronous flow (retrofit later) |
| D6 | `{file_type}_{file_id}_{ts}` filename at transfer | Bronze recovers `file_id` from `_metadata.file_name` with zero joins (Phase 3 pushback + reconciliation) | Lookup table join bronze↔Mongo |
| D7 | FastAPI as sole writer | One validation/audit/authorization surface; validators + `$jsonSchema` enforced in one place | React/direct Mongo writes (security + consistency holes) |
| D8 | `WARN` severity as shadow mode | New rules run without quarantining anything — observe, then flip to `ERROR` | Test rules in a separate environment only |
| D9 | Append-only `audit_log` + `file_lifecycle` | Immutable governance trail; "where has this file ever been" is answerable | Status-only tracking (history lost on update) |
| D10 | Purge gates on reconciliation (`MATCHED`) | No file is ever deleted before Databricks provably processed it | Age-based purge alone (can lose unprocessed files) |

## 7. Security & Governance

### Access model

| Principal | Access | Notes |
| --- | --- | --- |
| FastAPI app user | read-write on `fvr` (all collections + GridFS) | Least privilege; no `dbAdmin` |
| Analytics/read user | read-only on selected collections | Dashboards, exports, investigations |
| Admin actions | via FastAPI only, behind `ADMIN` role + approver | No human connects with write credentials |

* **RBAC (application layer):** `UPLOADER` (upload screens), `OPS` (status board, DEAD re-drive, re-classification), `ADMIN` (config CRUD + approvals). Roles cached in `users`, snapshot-frozen at upload time in `upload_sessions`.
* **Secrets:** SFTP/SharePoint credentials live in a vault — **never** in `source_systems.connection_ref` (that field holds a vault reference/path only).
* **PII posture:** uploaded file *contents* are never parsed into MongoDB — validation stores header names and sampled evidence only; PII classification lives in `schema_versions.fields[].pii_classification` and actual masking happens downstream in Databricks governed tagging. Optional field-level encryption for sensitive config fields (e.g. `identification_evidence`) if compliance requires.
* **Transport/storage:** TLS in transit; encryption at rest (default on Atlas / enable KMIP on self-managed).
* **Audit:** `audit_log` is append-only — enforced by giving the app user insert-only on that collection; every config change records actor, before/after, and `approver`.

## 8. Availability, DR & Scalability

### Availability
* 3-node replica set; automatic failover to a secondary.
* FastAPI event worker persists the **change-stream resume token** after each processed event — restart continues exactly where it stopped (no lost, no double-processed events).
* Transfer failures retry with exponential backoff (`RETRYING`), then surface on the ops board — never silently dropped.

### Backup / DR targets
| Objective | Target | Mechanism |
| --- | --- | --- |
| RPO | ≤ 15 min | Continuous oplog / PITR backups (Atlas) or frequent snapshots |
| RTO | ≤ 1 h | Restore snapshot + replay oplog; GridFS restore from the same backup |

* GridFS is inside the same database → metadata + bytes are restored **consistently together**.

### Scalability
* **Reads:** ops/analytics queries run with `secondaryPreferred` to keep the primary free for writes.
* **Sharding (future, on triggers):** `event_log` if it exceeds ~50 GB retained → shard key `{processing_status, received_at}` (TTL keeps it bounded first); `file_registry` beyond ~50M docs → hash-shard on `file_id`. Not enabled at launch — premature sharding hurts more than it helps.
* **Index budget:** every collection's indexes are enumerated in the LLD; review after the first load test (checkpoint R2).

## 9. Data Lifecycle & Integration Points

### File data lifecycle

```
RECEIVED → (GridFS staged, sha256) → VALIDATED → ROUTED → INTAKE_VOLUME/QUARANTINE (copy exists in volume)
→ BRONZE_PROCESSED (Phase 3) → RECONCILED MATCHED (resilience certificate)
→ ARCHIVE (cold copy) → PURGED (GridFS copy deleted — only after MATCHED + retention window)
```

### Integration points
| System | Direction | Interface |
| --- | --- | --- |
| React Portal | → FastAPI | REST/JSON (uploads chunked, multipart) — never touches Mongo |
| Transfer worker | MongoDB → UC volumes | GridFS download → volume write → checksum verify → `file_transfers` update |
| Databricks pushback | Databricks → FastAPI | `POST /internal/ingestion-results` (service principal, allowlisted) → `databricks_ingestion_results` |
| Watchers (Phase 2) | SharePoint/SFTP/Volume → FastAPI | Webhooks + pollers publish to `event_log` via internal endpoints |

## 10. HLD Open Items

1. Atlas vs self-managed replica set (drives backup/PITR + encryption story)
2. `event_log` TTL window — 30 days proposed; compliance confirmation needed
3. Field-level encryption scope for sensitive config (if any)
4. Shard triggers + budget approval (currently: none at launch)
5. Size of the analytics read-only user's surface (which collections are exposed to reporting)